# C3 — Beta sweep: does the Laplace NLL's reweighting help calibration?

`024_training_beta_sweep.ipynb` trained `unet_nll` at `beta = {0.0, 0.25, 0.75}`, holding
everything else (dataset, split, seed, optimizer, `laplace_nll` loss) fixed at the Round 1/2
recipe — `beta = 0.5` is not retrained there, it is `021_training_nll.ipynb`'s existing
checkpoint (`models/nll/unet_nll/`), the value every other NLL architecture in this project
still trains at.

This is the notebook `024` said "decides nothing" and pointed here for: the same machinery
as `C0_v1_v2.ipynb`/`C1_v1_v2_beta.ipynb` (fidelity, detection, coherence, §8's calibration
block), but comparing **four betas of one architecture** instead of two generations across
several. Unlike `C1` (Gaussian `v1` vs. Laplace `v2` — two different densities, `nll` marked
not comparable), every run here is the same Laplace NLL at the same architecture, so `nll`
**is** directly comparable across columns this time — only `beta` moves.

**What `beta` does** (`scripts/losses.laplace_nll_loss`, `fixing.md` #10): the loss is
`stop_gradient(b)^beta * (|y - mu| / b + log b)`. `beta = 0` is the plain Laplace NLL
(high-uncertainty regions downweighted most on the gradient reaching `mu`); `beta = 1` removes
the weighting entirely. The grid brackets the project's default (`0.5`) on both sides.

**Read this per `024`'s own warning**: rank on fidelity (§5) and calibration (§8) —
`z_std` toward `1.0`, `ence` down, `error_sigma_spearman` up, coverage near nominal. Detection
AUROC (§6) is included for completeness but `C0`+`C1` found it correlates with *worse*
reconstruction across 16 architecture x generation points (Spearman `rho = +0.665`,
`p = 0.005`) — a confound `024` explicitly flags as unresolved, so don't let §6 alone decide
anything here either.

Companion notebooks: `024_training_beta_sweep.ipynb` (trains the three new points),
`021_training_nll.ipynb` (the `beta = 0.5` reference), `C0_v1_v2.ipynb`/`C1_v1_v2_beta.ipynb`
(the generation comparison this reuses the machinery from).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check. Same building blocks as `C0`/`C1`; `scripts.trainer` (the deterministic loader) is not needed here — every column is `unet_nll`.

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.calibration import (
    evaluate_calibration,
    laplace_sigma_from_scale,
    learned_zscore,
    structural_zscore,
)
from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import DEFAULT_Z_SCALE, plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. The `data/test/` images

Every `(rgb, ir)` pair under `data/test/`, discovered generically — same as `040`/`C0`/`C1`.
The three with a hand-drawn mask (`GT01`/`GT02`/`GT03`) additionally carry a detection ground
truth; fidelity/coherence/calibration need no mask and run on all of them.

In [ ]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) + sorted(TEST_RGB_DIR.glob("*.png"))
image_pairs = [
    (p, TEST_IR_DIR / p.name)
    for p in sorted(rgb_paths)
    if (TEST_IR_DIR / p.name).exists()
]
gt_stems = {
    p.name.removesuffix("_Map.png") for p in sorted(ANNOTATIONS_DIR.glob("*_Map.png"))
}
gt_stems &= {p.stem for p, _ in image_pairs}

print(f"data/test/ images: {len(image_pairs)} | {[p.stem for p, _ in image_pairs]}")
print(f"with a ground-truth mask: {sorted(gt_stems)}")

if not image_pairs:
    raise RuntimeError("No (rgb, ir) pairs found under data/test/.")

## 2. The beta grid, and where each checkpoint lives

`BETA_RUNS` mirrors `024`'s layout: `beta = 0.5` is the standing reference
(`models/nll/unet_nll/`, trained by `021`, never retrained here), the other three come from
`024`'s `models/beta_sweep/beta_<value>/unet_nll/`. A missing checkpoint (e.g. `024` has not
been run yet) is skipped with a message, not an error.

Every column is the same architecture and the same Laplace distribution — `sigma_source` does
not vary here, unlike `C0`/`C1` where it dispatched on generation.

In [ ]:
ARCH = "unet_nll"
FAMILY = "nll"
LOSS_NAME = "laplace_nll"
REFERENCE_BETA = settings.NLL_BETA  # 0.5, the project-wide default

BETA_RUNS = {
    0.0: settings.MODELS_DIR / "beta_sweep" / "beta_0.00",
    0.25: settings.MODELS_DIR / "beta_sweep" / "beta_0.25",
    REFERENCE_BETA: settings.MODELS_DIR / "nll",
    0.75: settings.MODELS_DIR / "beta_sweep" / "beta_0.75",
}

SIGNAL_KINDS = ("raw delta", "structural delta", "confidence")
NLL_SIGNAL_KINDS = ("|z| (raw/sigma)", "structural z")

missing = [
    beta
    for beta, model_dir in BETA_RUNS.items()
    if not (model_dir / ARCH / "best_model.keras").exists()
]
print(f"Betas to compare: {sorted(BETA_RUNS)}")
print(f"Missing checkpoints (will be skipped): {missing or 'none'}")

## 3. Signals, per beta

Identical construction to `040`/`041`/`C0`/`C1` — **raw delta**, **structural delta**,
**confidence**, plus **`\|z\|`** and **structural z**, all read off the same
`sigma = laplace_sigma_from_scale(exp(log_b))` conversion for every beta (unlike `C0`/`C1`,
there is only one distribution here, so no per-column dispatch is needed).

In [ ]:
MASK_THRESHOLD = 127  # midpoint threshold for the mask's anti-aliased edges


def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load an ``(rgb, ir)`` pair as float arrays in ``[0, 1]``."""
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    """Load a hand-drawn ground-truth mask as a boolean array."""
    mask_path = ANNOTATIONS_DIR / f"{stem}_Map.png"
    return np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD


def fidelity(ir: np.ndarray, mu: np.ndarray) -> dict[str, float]:
    """MAE/SSIM/PSNR of a prediction against the real IR, as in 030."""
    real = tf.constant(ir[..., np.newaxis])
    pred = tf.constant(mu[..., np.newaxis])
    return {
        "mae": float(np.mean(np.abs(ir - mu))),
        "ssim": float(tf.image.ssim(real, pred, max_val=1.0)),
        "psnr": float(tf.image.psnr(real, pred, max_val=1.0)),
    }


def predict_signals(
    model: tf.keras.Model, rgb: np.ndarray, ir: np.ndarray
) -> tuple[dict[str, float], dict[str, np.ndarray], np.ndarray, np.ndarray]:
    """Predict one image and build every signal this architecture supports."""
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = ir.shape
    pred = model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, :]

    mu = pred[..., 0]
    sigma = laplace_sigma_from_scale(np.exp(pred[..., 1]))
    result = analyze_delta(ir, mu)
    signals = {
        "raw delta": result.raw_delta,
        "structural delta": result.structural_delta,
        "confidence": result.confidence_map,
        "|z| (raw/sigma)": np.abs(learned_zscore(ir, mu, sigma)),
        "structural z": structural_zscore(result.structural_delta, sigma),
    }
    return fidelity(ir, mu), signals, sigma, mu

## 4. The sweep

One beta at a time, loaded, scored over every `data/test/` image, then released before the
next — same reasoning as `C0`/`C1`'s memory handling.

`PLOT_IMAGES`/`PLOT_SIGNALS` control only the figures; the tables in §5-§8 always cover every
image/signal.

In [ ]:
PLOT_IMAGES = set()  # -> set(gt_stems) to plot the ground-truth images
PLOT_SIGNALS = set()  # -> ("raw delta", "structural delta") to plot those

fidelity_rows: dict[float, dict[str, list[float]]] = {}
coherence_rows: dict[tuple[float, str], list[float]] = {}
auroc_rows: dict[tuple[float, str], dict[str, float]] = {}
ap_rows: dict[tuple[float, str], dict[str, float]] = {}
calibration_rows: dict[float, dict[str, list[float]]] = {}

compared_betas: list[float] = []

for beta, model_dir in BETA_RUNS.items():
    try:
        model = load_model_nll(
            ARCH, model_dir=model_dir, loss_name=LOSS_NAME, beta=beta
        )
    except FileNotFoundError as exc:
        print(f"[skip] beta={beta}: {exc}")
        continue

    compared_betas.append(beta)
    tag = " (reference, from 021)" if beta == REFERENCE_BETA else ""
    print(f"\n=== beta={beta}{tag} ===")

    for rgb_path, ir_path in image_pairs:
        stem = rgb_path.stem
        rgb, ir = load_pair(rgb_path, ir_path)
        mask = load_mask(stem) if stem in gt_stems else None

        scores, signals, sigma, mu = predict_signals(model, rgb, ir)

        for metric, value in scores.items():
            fidelity_rows.setdefault(beta, {}).setdefault(metric, []).append(value)

        for kind, signal in signals.items():
            coherence_rows.setdefault((beta, kind), []).append(
                stroke_coherence(signal).coherence
            )
            if mask is not None:
                detection = evaluate_detection(signal, mask)
                auroc_rows.setdefault((beta, kind), {})[stem] = detection.auroc
                ap_rows.setdefault((beta, kind), {})[stem] = detection.average_precision

        summary = evaluate_calibration(ir, mu, sigma, distribution="laplace").summary()
        for metric, value in summary.items():
            calibration_rows.setdefault(beta, {}).setdefault(metric, []).append(value)

        if stem in PLOT_IMAGES:
            for kind in PLOT_SIGNALS:
                fig = plot_signal_comparison(
                    ir, {f"beta={beta}": signals[kind]},
                    title=f"{ARCH} — {kind} — {stem} — beta={beta}",
                    vrange=(0.0, 1.0),
                )
                plt.show()
                plt.close(fig)
            for kind in NLL_SIGNAL_KINDS:
                scaled, vrange = DEFAULT_Z_SCALE.apply_many({f"beta={beta}": signals[kind]})
                fig = plot_signal_comparison(
                    ir, scaled, title=f"{ARCH} — {kind} — {stem} — beta={beta}", vrange=vrange
                )
                plt.show()
                plt.close(fig)

        print(f"  {stem}: scored")
        del signals, rgb, ir, mask

    del model
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nCompared: {sorted(compared_betas)}")
if not compared_betas:
    raise RuntimeError("No beta had a checkpoint to load — nothing to compare.")

## 5. Reconstruction fidelity — how well each beta predicts the IR

`mae`/`ssim`/`psnr` of `mu` against the real IR, averaged over all `data/test/` images —
the same axis `030_evaluation.ipynb` reports. `024` warned `val_loss` is not comparable
across beta (the reweighting scales the loss itself); these three metrics do not carry the
beta weight and are the first honest read on which beta trained best on fidelity alone.

`delta` is against the `beta = 0.5` reference (positive = better than the current default),
oriented per metric so positive always means "improvement."

In [ ]:
LOWER_IS_BETTER = {"mae": True, "ssim": False, "psnr": False}


def gain(metric: str, ref: float, other: float) -> float:
    return ref - other if LOWER_IS_BETTER[metric] else other - ref


col_w = 12
header = "metric".ljust(10)
for beta in sorted(compared_betas):
    header += f"beta={beta}".rjust(col_w)
print(header)
print("-" * len(header))

for metric in LOWER_IS_BETTER:
    row = metric.ljust(10)
    ref_value = float(np.mean(fidelity_rows[REFERENCE_BETA][metric]))
    for beta in sorted(compared_betas):
        value = float(np.mean(fidelity_rows[beta][metric]))
        marker = "" if beta == REFERENCE_BETA else f" ({gain(metric, ref_value, value):+.4f})"
        row += f"{value:.4f}{marker}".rjust(col_w)
    print(row)

print(f"\n(mean over {len(image_pairs)} data/test/ images; "
      f"parenthesized delta vs. beta={REFERENCE_BETA}, positive = improvement)")

## 6. Detection — read with `024`'s confound warning in mind

AUROC of each signal against the hand-drawn masks, averaged over `GT01`/`GT02`/`GT03` only.
`0.5` is chance. **`C0`+`C1` found AUROC correlates with *worse* reconstruction fidelity
across 16 architecture x generation points** — a confound not yet ruled out for beta, so a
beta that wins here and loses in §5 is not obviously the better choice. With `n = 3` the mean
also hides a lot; the per-image breakdown follows.

In [ ]:
def detection_table(rows: dict[tuple[float, str], dict[str, float]], label: str) -> None:
    kinds_seen = {kind for _, kind in rows}
    ordered_kinds = [k for k in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS) if k in kinds_seen]
    name_w = 24
    header = "signal".ljust(name_w)
    for beta in sorted(compared_betas):
        header += f"beta={beta}".rjust(12)
    print(header)
    print("-" * len(header))
    for kind in ordered_kinds:
        if not all((beta, kind) in rows for beta in compared_betas):
            continue
        row = kind.ljust(name_w)
        for beta in sorted(compared_betas):
            value = float(np.mean(list(rows[(beta, kind)].values())))
            row += f"{value:.4f}".rjust(12)
        print(row)
    print(f"\n({label}, mean over {sorted(gt_stems)})")


if auroc_rows:
    print("AUROC\n")
    detection_table(auroc_rows, "AUROC")
    print("\n\naverage precision\n")
    detection_table(ap_rows, "average precision")
else:
    print("No ground-truth mask found under data/test/annotations/ — section skipped.")

### Per-image AUROC breakdown

The `n = 3` mean above can be carried entirely by one favourable image.

In [ ]:
if auroc_rows:
    stems = sorted(gt_stems)
    name_w = 24
    header = "signal".ljust(name_w)
    for stem in stems:
        for beta in sorted(compared_betas):
            header += f"{stem} b={beta}".rjust(14)
    print(header)
    print("-" * len(header))
    for kind in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS):
        if not all((beta, kind) in auroc_rows for beta in compared_betas):
            continue
        row = kind.ljust(name_w)
        for stem in stems:
            for beta in sorted(compared_betas):
                row += f"{auroc_rows[(beta, kind)][stem]:.3f}".rjust(14)
        print(row)
else:
    print("No ground-truth mask found — section skipped.")

## 7. Stroke coherence — the reference-free corroboration

How much of each signal is oriented, line-like structure rather than isotropic noise
(`scripts.stroke_stats`) — no mask needed, so it runs on all `data/test/` images and is a
structurally independent check on the AUROC ranking above.

In [ ]:
name_w = 24
header = "signal".ljust(name_w)
for beta in sorted(compared_betas):
    header += f"beta={beta}".rjust(18)
print(header)
print("-" * len(header))

for kind in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS):
    if not all((beta, kind) in coherence_rows for beta in compared_betas):
        continue
    row = kind.ljust(name_w)
    for beta in sorted(compared_betas):
        values = np.array(coherence_rows[(beta, kind)])
        row += f"{values.mean():.3f}±{values.std():.3f}".rjust(18)
    print(row)

print(f"\n(mean ± std over {len(image_pairs)} data/test/ images)")

## 8. Sigma calibration — the axis this sweep is actually about

`024`'s own framing: no training-time metric measures sigma behaviour, so this is where the
beta question gets settled. `nll` **is** comparable across these columns (same Laplace
density throughout, unlike `C1`'s Gaussian-vs-Laplace) but is not the target — beta is
constructed to trade `nll` for weighting, so a lower `nll` at low beta is expected and not by
itself an improvement. Judge on `z_std` (toward `1.0`), `ence` (down), `error_sigma_spearman`
(up — sigma should track where the model is actually wrong), coverage (near nominal
`0.6827`/`0.9545`), and `dispersion` (a large *constant* sigma scores well on coverage and is
useless — this catches that).

In [ ]:
CALIB_KEYS = [
    "nll",
    "ence",
    "z_std",
    "coverage_1s",
    "coverage_2s",
    "error_sigma_spearman",
    "sharpness",
    "dispersion",
]

name_w = 24
header = "metric".ljust(name_w)
for beta in sorted(compared_betas):
    header += f"beta={beta}".rjust(14)
print(header)
print("-" * len(header))

for metric in CALIB_KEYS:
    row = metric.ljust(name_w)
    for beta in sorted(compared_betas):
        value = float(np.mean(calibration_rows[beta][metric]))
        row += f"{value:.4f}".rjust(14)
    print(row)

print(f"\n(mean over {len(image_pairs)} data/test/ images)")
print("nominal coverage: 1s = 0.6827, 2s = 0.9545; calibrated z_std = 1.0")
print(f"reference column: beta={REFERENCE_BETA} (the project-wide default, from 021)")

## 9. Verdict — which beta actually moved anything

A count, per beta, of how many measured quantities improved **relative to the `beta = 0.5`
reference**. Deliberately coarse: it collapses fidelity, detection and coherence, which do
not have to agree, and §6 carries the confound flagged above — read §5 and §8 before trusting
this table over them.

In [ ]:
def tally(gains: list[float]) -> str:
    """``"k/n"`` — how many of these gains are improvements over the reference."""
    if not gains:
        return "-"
    return f"{sum(g > 0 for g in gains)}/{len(gains)}"


ALL_KINDS = (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS)

name_w = 12
print(
    "beta".ljust(name_w)
    + "fidelity".rjust(12)
    + "detection".rjust(12)
    + "coherence".rjust(12)
    + "calibration".rjust(14)
)
print("-" * (name_w + 50))

CALIB_HIGHER_IS_BETTER = {"z_std": False, "ence": False, "error_sigma_spearman": True}

for beta in sorted(compared_betas):
    if beta == REFERENCE_BETA:
        print(f"{str(beta).ljust(name_w)}{'(reference)'.rjust(62)}")
        continue

    fidelity_gains = [
        gain(
            metric,
            float(np.mean(fidelity_rows[REFERENCE_BETA][metric])),
            float(np.mean(fidelity_rows[beta][metric])),
        )
        for metric in LOWER_IS_BETTER
    ]
    detection_gains = [
        float(np.mean(list(auroc_rows[(beta, kind)].values())))
        - float(np.mean(list(auroc_rows[(REFERENCE_BETA, kind)].values())))
        for kind in ALL_KINDS
        if (beta, kind) in auroc_rows and (REFERENCE_BETA, kind) in auroc_rows
    ]
    coherence_gains = [
        float(np.mean(coherence_rows[(beta, kind)]))
        - float(np.mean(coherence_rows[(REFERENCE_BETA, kind)]))
        for kind in ALL_KINDS
        if (beta, kind) in coherence_rows
    ]
    calibration_gains = [
        (
            float(np.mean(calibration_rows[REFERENCE_BETA][metric]))
            - float(np.mean(calibration_rows[beta][metric]))
            if not higher_is_better
            else float(np.mean(calibration_rows[beta][metric]))
            - float(np.mean(calibration_rows[REFERENCE_BETA][metric]))
        )
        for metric, higher_is_better in CALIB_HIGHER_IS_BETTER.items()
    ]

    print(
        str(beta).ljust(name_w)
        + tally(fidelity_gains).rjust(12)
        + tally(detection_gains).rjust(12)
        + tally(coherence_gains).rjust(12)
        + tally(calibration_gains).rjust(14)
    )

print(f"\n(k/n better than beta={REFERENCE_BETA}; calibration = z_std/ence/error_sigma_spearman)")